# NPUFlow training agent

Turns this Kaggle GPU session into a ClearML worker for the queue `kaggle`. The logic lives in
[`kaggle/agent.py`](https://github.com/ryngach/NPUFlow/blob/main/kaggle/agent.py) of the repository and is
downloaded on every run, so this notebook itself does not need to change.

**Setup, once:** *Add-ons → Secrets*, add `CLEARML_API_ACCESS_KEY` and `CLEARML_API_SECRET_KEY`
(ClearML: *Settings → Workspace → Create new credentials*) and tick both so they are attached to this notebook.

**To start a session:** *Save Version → Save & Run All (Commit)* with the GPU accelerator.
Secrets are only available to runs started from this editor, not to versions pushed through the Kaggle API.


In [ ]:
import os, subprocess, sys, time

os.environ["NPUFLOW_SESSION_START"] = str(time.time())
missing = []
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
        except Exception:
            missing.append(_name)
except Exception as e:
    missing = ["CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"]
    print("kaggle_secrets is not available:", e)

if missing:
    print("Secrets missing or not attached to this notebook:", missing)
    print("Attach them under Add-ons -> Secrets and start the run from this editor (Save Version).")
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ryngach/NPUFlow.git", "/kaggle/working/NPUFlow"], check=True)
    subprocess.run([sys.executable, "/kaggle/working/NPUFlow/kaggle/agent.py"], check=True)
